In [2]:
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

In [3]:
print(os.getcwd())

C:\Users\pockyc\AppData\Local\Programs\Microsoft VS Code


In [4]:
from Connect4env import Connect4Env
from DQN.Agent import DQNAgent

ModuleNotFoundError: No module named 'Connect4env'

In [5]:
env = Connect4Env()

NameError: name 'Connect4Env' is not defined

In [ ]:
p1agent = DQNAgent(1)
p2agent = DQNAgent(-1)

p1agent.replay_buffer.buffer.clear()
p2agent.replay_buffer.buffer.clear()

In [ ]:
num_episodes = 10000
batch_size = 64

In [ ]:
save_dir = "Saved_models"
os.makedirs(save_dir, exist_ok=True)

for episode in range(num_episodes):
    state, _ = env.reset()
    done = False
    
    # Logging
    p1_reward = 0
    p2_reward = 0
    invalid_moves = 0
    step_count = 0
    winner = "None"

    #  unified last move tracking
    last = {
        1: {"state": None, "action": None, "valid_actions": None},
        -1: {"state": None, "action": None, "valid_actions": None}
    }

    while not done:
        
        current_player = env.current_player
        agent = p1agent if current_player == 1 else p2agent

        valid_actions = [c for c in range(env.ncol) if env.game.is_valid_move(c)]
        action = agent.select_action(state, valid_actions)

        #  Save current move (for delayed storage)
        last[current_player] = {
            "state": state,
            "action": action,
            "valid_actions": valid_actions
        }

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        next_valid_actions = [c for c in range(env.ncol) if env.game.is_valid_move(c)]

        # Track invalid move
        if reward == -10.0:
            invalid_moves += 1

        # Logging rewards
        if current_player == 1:
            p1_reward += reward
        else:
            p2_reward += reward

        # ==================================================
        #  DELAYED TRANSITION STORAGE
        # ==================================================
        player_just_acted = current_player

        if last[player_just_acted]["state"] is not None:
            agent.replay_buffer.push(
                last[player_just_acted]["state"],
                last[player_just_acted]["action"],
                reward,
                next_state,
                done,
                last[player_just_acted]["valid_actions"]
            )

        # ==================================================
        #  TERMINAL HANDLING (WIN / LOSS PROPAGATION)
        # ==================================================
        if done:
            player = player_just_acted
            if reward > 0:  # someone won
                if player == 1:
                    p2_reward -= reward  
                else:
                    p1_reward -= reward  
            agent = p1agent if player == 1 else p2agent

            # Store final transition (winner or last mover)
            agent.replay_buffer.push(
                state,
                action,
                reward,
                next_state,
                done,
                valid_actions
            )

            # Opponent gets opposite reward
            opponent = -player
            opponent_agent = p1agent if opponent == 1 else p2agent

            if last[opponent]["state"] is not None:
                opponent_agent.replay_buffer.push(
                    last[opponent]["state"],
                    last[opponent]["action"],
                    -reward,
                    next_state,
                    done,
                    last[opponent]["valid_actions"]
                )

            # Winner logging
            if reward > 0:
                winner = "P1" if player == 1 else "P2"
            elif reward < 0:
                winner = "Invalid Move"
            else:
                winner = "Draw"

        state = next_state
        step_count += 1

        # ==================================================
        # 🔴 TRAIN BOTH AGENTS
        # ==================================================
        if len(p1agent.replay_buffer.buffer) > p1agent.batch_size:
            p1agent.train()

        if len(p2agent.replay_buffer.buffer) > p2agent.batch_size:
            p2agent.train()

    # =========================
    # LOGGING
    # =========================
    print(f"""
Episode {episode}
Steps: {step_count}
Winner: {winner}
P1 Total Reward: {p1_reward:.2f}
P2 Total Reward: {p2_reward:.2f}
Invalid Moves: {invalid_moves}
""")

    # ================================
    # SAVE MODELS EVERY 1000 EPISODES
    # ================================
    if (episode + 1) % 1000 == 0:
        ep = episode + 1

        torch.save(
            p1agent.online_net.state_dict(),
            f"{save_dir}/P1_episode{ep}.pth"
        )

        torch.save(
            p2agent.online_net.state_dict(),
            f"{save_dir}/P2_episode{ep}.pth"
        )

        print(f"✅ Saved models at episode {ep}")